# From RAG To Knowledge Graphs In 2027: Neo4j Demo
Notebook-style Python cells for manual copy-paste into Jupyter.

## Setup
Fill in `.env` before running the extraction or Neo4j cells.

In [ ]:
import os
import json
import runpy
from pathlib import Path
from typing import Any

import frontmatter
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from pydantic import BaseModel, ConfigDict, Field, field_validator
from pydantic_ai import Agent

In [ ]:
load_dotenv()

In [ ]:
PROJECT_ROOT = Path.cwd()
INPUT_PATHS = sorted(PROJECT_ROOT.glob("data/*.md"))
PROCESSED_DIR = PROJECT_ROOT / "processed"
JSONL_PATH = PROCESSED_DIR / "graph_extractions.jsonl"

MODEL_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT", "openai:gpt-4.1-mini")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "<instance-id>")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "<password>")
NEO4J_URI = os.getenv("NEO4J_URI", f"neo4j+s://{NEO4J_USERNAME}.databases.neo4j.io")

print(f"Input files: {len(INPUT_PATHS)}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Neo4j URI: {NEO4J_URI}")

In [ ]:
if not INPUT_PATHS:
    raise RuntimeError("No markdown files found in data/*.md")

INPUT_PATHS[:10]

## Inspect The Corpus
Each file should have frontmatter like `title`, `source`, `date_published`, `author`, and `url`.


In [ ]:
sample_path = INPUT_PATHS[0]
sample_post = frontmatter.load(sample_path)

print(f"Path: ${sample_path}")
print(sample_post.metadata)
print()
print(sample_post.content[:1200])

## Define The Extraction Schema
`source_type` is removed. It does not change extraction quality, ingestion, or the demo queries.

In [ ]:
class DocumentGraphExtraction(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    summary: str = Field(description="A concise 2-4 sentence summary.")
    technologies: list[str] = Field(default_factory=list)
    concepts: list[str] = Field(default_factory=list)
    business_topics: list[str] = Field(default_factory=list)
    people: list[str] = Field(default_factory=list)
    organizations: list[str] = Field(default_factory=list)

    @field_validator(
        "technologies", "concepts", "business_topics", "people", "organizations", mode="before"
    )
    @classmethod
    def ensure_list(cls, value: Any) -> list[str]:
        if value is None:
            return []
        if isinstance(value, str):
            return [value]
        return list(value)

    @field_validator("technologies", "concepts", "business_topics", "people", "organizations")
    @classmethod
    def dedupe_and_clean(cls, values: list[str]) -> list[str]:
        seen: set[str] = set()
        cleaned_values: list[str] = []
        for value in values:
            cleaned = " ".join(str(value).split()).strip()
            if not cleaned or cleaned.casefold() in seen:
                continue
            seen.add(cleaned.casefold())
            cleaned_values.append(cleaned)
        return cleaned_values


DocumentGraphExtraction.model_json_schema()


## Load The Prompt
The prompt lives in `prompts/graph-metadata-extractor.py`, as the plan required.


In [ ]:
GRAPH_METADATA_EXTRACTOR = runpy.run_path(
    str(PROJECT_ROOT / "prompts" / "graph-metadata-extractor.py")
)["GRAPH_METADATA_EXTRACTOR"]

print(GRAPH_METADATA_EXTRACTOR)

## Create The Extraction Agent

In [ ]:
settings = {"max_tokens": 4096}

In [ ]:
# from pydantic_ai.models.openai import OpenAIChatModel
# from pydantic_ai.providers.azure import AzureProvider


# llm = OpenAIChatModel(
#     model_name=MODEL_NAME,
#     provider=AzureProvider(
#         azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
#         api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
#         api_key=os.getenv("AZURE_OPENAI_API_KEY"),
#     ),
#     settings=settings,
# )

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

llm = OpenAIChatModel(
    "openai/gpt-5-mini-2025-08-07",
    provider=OpenAIProvider(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY")
    ),
)


In [ ]:
extraction_agent = Agent(
    llm,
    instructions=GRAPH_METADATA_EXTRACTOR,
    output_type=DocumentGraphExtraction,
)

## Dry-run extraction (1 document)
This is the low-risk live path.

In [ ]:
# one_result = await extraction_agent.run(
#     f"""
#     Extract graph-friendly metadata from this markdown document body.

#     <document_body>
#     {sample_post.content}
#     </document_body>
#     """.strip()
# )

In [ ]:
# one_result.output

## Run Extraction Across `data/*.md`

### Precompute Extractions To JSONL
One line per document so the batch can be reloaded later without rerunning extraction.


In [ ]:
extractions: list[dict[str, Any]] = []
PROCESSED_DIR.mkdir(exist_ok=True)

with JSONL_PATH.open("a", encoding="utf-8") as jsonl_file:
    for path in INPUT_PATHS:
        post = frontmatter.load(path)
        result = await extraction_agent.run(
            f"""
            Extract graph-friendly metadata from this markdown document body.

            <document_body>
            {post.content}
            </document_body>
            """.strip()
        )
        extraction = result.output
        record = {
            "source_path": str(path.relative_to(PROJECT_ROOT)),
            "document_id": post.metadata.get("url") or str(path.relative_to(PROJECT_ROOT)),
            "metadata": dict(post.metadata),
            "graph": extraction.model_dump(mode="json"),
        }
        jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        jsonl_file.flush()
        extractions.append({"path": path, "post": post, "extraction": extraction})

print(f"Wrote {len(extractions)} JSONL records to {JSONL_PATH}")

In [ ]:
len(extractions)

In [ ]:
# Optional quick reload check
with JSONL_PATH.open("r", encoding="utf-8") as f:
    preview = json.loads(next(f))

preview

In [ ]:
pd.DataFrame(
    [
        {
            "path": str(item["path"]),
            "technologies": item["extraction"].technologies,
            "concepts": item["extraction"].concepts,
            "business_topics": item["extraction"].business_topics,
            "people": item["extraction"].people,
            "organizations": item["extraction"].organizations,
        }
        for item in extractions
    ]
)

## Write Enriched Outputs
The processed files keep the original body and add `graph` data to frontmatter.


In [ ]:
PROCESSED_DIR.mkdir(exist_ok=True)

for item in extractions:
    path = item["path"]
    post = item["post"]
    extraction = item["extraction"]
    metadata = dict(post.metadata)
    metadata["document_id"] = metadata.get("url") or str(path)
    metadata["graph"] = extraction.model_dump(mode="json")
    frontmatter.dump(frontmatter.Post(post.content, **metadata), PROCESSED_DIR / path.name)

print(f"Wrote {len(extractions)} processed markdown files to {PROCESSED_DIR}")


## Ontology
Labels: `Document`, `Source`, `Technology`, `Concept`, `BusinessTopic`, `Person`, `Organization`.
Relationships: `FROM_SOURCE`, `MENTIONS_TECHNOLOGY`, `MENTIONS_CONCEPT`, `MENTIONS_BUSINESS_TOPIC`, `MENTIONS_PERSON`, `MENTIONS_ORGANIZATION`.


### Connect To Neo4j Aura
This matches the current driver docs and does not set a separate database name.


In [ ]:
from neo4j import GraphDatabase, Result, RoutingControl

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()


def cypher_write(query: str, **params: Any):
    return driver.execute_query(
        query,
        routing_=RoutingControl.WRITE,
        **params,
    )


def cypher_df(query: str, **params: Any):
    return driver.execute_query(
        query,
        routing_=RoutingControl.READ,
        result_transformer_=Result.to_df,
        **params,
    )



## Create Constraints

In [ ]:
constraint_queries = [
    "CREATE CONSTRAINT document_document_id IF NOT EXISTS FOR (d:Document) REQUIRE d.document_id IS UNIQUE",
    "CREATE CONSTRAINT source_name IF NOT EXISTS FOR (s:Source) REQUIRE s.name IS UNIQUE",
    "CREATE CONSTRAINT technology_name IF NOT EXISTS FOR (n:Technology) REQUIRE n.name IS UNIQUE",
    "CREATE CONSTRAINT concept_name IF NOT EXISTS FOR (n:Concept) REQUIRE n.name IS UNIQUE",
    "CREATE CONSTRAINT business_topic_name IF NOT EXISTS FOR (n:BusinessTopic) REQUIRE n.name IS UNIQUE",
    "CREATE CONSTRAINT person_name IF NOT EXISTS FOR (n:Person) REQUIRE n.name IS UNIQUE",
    "CREATE CONSTRAINT organization_name IF NOT EXISTS FOR (n:Organization) REQUIRE n.name IS UNIQUE",
]

for query in constraint_queries:
    cypher_write(query)

## Ingest Documents

In [ ]:
entity_specs = [
    ("technologies", "Technology", "MENTIONS_TECHNOLOGY"),
    ("concepts", "Concept", "MENTIONS_CONCEPT"),
    ("business_topics", "BusinessTopic", "MENTIONS_BUSINESS_TOPIC"),
    ("people", "Person", "MENTIONS_PERSON"),
    ("organizations", "Organization", "MENTIONS_ORGANIZATION"),
]

for path in processed_paths:
    post = frontmatter.load(path)
    metadata = dict(post.metadata)
    graph = metadata["graph"]
    document_id = metadata.get("document_id") or metadata.get("url") or str(path)

    cypher_write(
        """
        MERGE (d:Document {document_id: $document_id})
        SET d.title = $title,
            d.author = $author,
            d.date_published = $date_published,
            d.url = $url,
            d.summary = $summary
        MERGE (s:Source {name: $source})
        MERGE (d)-[:FROM_SOURCE]->(s)
        """,
        document_id=document_id,
        title=metadata.get("title") or path.stem,
        author=metadata.get("author"),
        date_published=metadata.get("date_published"),
        url=metadata.get("url"),
        summary=graph["summary"],
        source=metadata.get("source") or "Unknown source",
    )
    for field_name, label, relationship in entity_specs:
        for name in graph[field_name]:
            cypher_write(
                f"""
                MATCH (d:Document {{document_id: $document_id}})
                MERGE (e:{label} {{name: $name}})
                MERGE (d)-[:{relationship}]->(e)
                """,
                document_id=document_id,
                name=name,
            )

print(f"Ingested {len(processed_paths)} documents.")

## Query 1: Documents And Sources


In [ ]:
query_1 = """
MATCH (d:Document)-[:FROM_SOURCE]->(s:Source)
RETURN d.title AS document, s.name AS source
ORDER BY source, document
LIMIT 25
"""

cypher_df(query_1)

## Query 2: Documents Mentioning One Entity

In [ ]:
entity_name = "ChatGPT"

query_2 = """
MATCH (d:Document)-[:MENTIONS_TECHNOLOGY|MENTIONS_CONCEPT]->(e)
WHERE e.name = $entity_name
RETURN e.name AS entity, d.title AS document
ORDER BY document
"""

cypher_df(query_2, entity_name=entity_name)

### Top technology across the corpus

In [ ]:
query_3 = """
MATCH (d:Document)-[:FROM_SOURCE]->(s:Source)
MATCH (d)-[:MENTIONS_TECHNOLOGY]->(b:Technology)
WITH b, collect(DISTINCT s.name) AS sources, count(DISTINCT d) AS document_count
WHERE size(sources) > 1
RETURN b.name AS business_topic, document_count, sources
ORDER BY document_count DESC, business_topic
LIMIT 20
"""

cypher_df(query_3)

### Top business topics across the corpus

In [ ]:
query_4 = """
MATCH (d:Document)-[:FROM_SOURCE]->(s:Source)
MATCH (d)-[:MENTIONS_BUSINESS_TOPIC]->(b:BusinessTopic)
WITH b, collect(DISTINCT s.name) AS sources, count(DISTINCT d) AS document_count
WHERE size(sources) > 1
RETURN b.name AS business_topic, document_count, sources
ORDER BY document_count DESC, business_topic
LIMIT 20
"""

cypher_df(query_4)

### Technologies connected to a business topic like product strategy


In [ ]:
_query = """
MATCH (d:Document)-[:MENTIONS_BUSINESS_TOPIC]->(b:BusinessTopic {name: $topic})
MATCH (d)-[:MENTIONS_TECHNOLOGY]->(t:Technology)
RETURN t.name AS technology, count(DISTINCT d) AS document_count
ORDER BY document_count DESC, technology
"""

cypher_df(_query, topic="product strategy")

### Business topics connected to a technology like ChatGPT


In [ ]:
_query = """
MATCH (d:Document)-[:MENTIONS_TECHNOLOGY]->(t:Technology {name: $technology})
MATCH (d)-[:MENTIONS_BUSINESS_TOPIC]->(b:BusinessTopic)
RETURN b.name AS business_topic, count(DISTINCT d) AS document_count
ORDER BY document_count DESC, business_topic
LIMIT 5
"""

cypher_df(_query, technology="ChatGPT")

### Pairs of technologies that co-occur


In [ ]:
_query = """
MATCH (d:Document)-[:MENTIONS_TECHNOLOGY]->(t1:Technology)
MATCH (d)-[:MENTIONS_TECHNOLOGY]->(t2:Technology)
WHERE t1.name < t2.name
RETURN t1.name AS technology_1, t2.name AS technology_2, count(DISTINCT d) AS co_mentions
ORDER BY co_mentions DESC, technology_1, technology_2
LIMIT 15
"""

cypher_df(_query)

## Query 4: Neighborhood Of One Document

In [ ]:
query_5 = """
MATCH (d:Document {document_id: $document_id})
MATCH (d)-[r1:MENTIONS_BUSINESS_TOPIC|MENTIONS_TECHNOLOGY]->(shared)
MATCH (other:Document)-[r2:MENTIONS_BUSINESS_TOPIC|MENTIONS_TECHNOLOGY]->(shared)
WHERE other <> d
WITH
    d,
    other,
    collect(DISTINCT shared.name) AS overlap,
    sum(
        CASE
            WHEN type(r1) = 'MENTIONS_BUSINESS_TOPIC' AND type(r2) = 'MENTIONS_BUSINESS_TOPIC' THEN 2
            ELSE 1
        END
    ) AS score
ORDER BY score DESC, other.title
RETURN
    d.title AS document,
    other.title AS similar_document,
    score,
    overlap[..8] AS shared_topics
LIMIT 2
"""
cypher_df(query_5, document_id=document_id)

In [ ]:
document_id = frontmatter.load(processed_paths[0]).metadata["document_id"]

query_6 = """
MATCH (d:Document {document_id: $document_id})
OPTIONAL MATCH (d)-[:FROM_SOURCE]->(s:Source)
OPTIONAL MATCH (d)-[r]->(e)
WHERE e:Technology OR e:Concept OR e:BusinessTopic OR e:Person OR e:Organization
RETURN d.title AS document, s.name AS source, type(r) AS relationship, e.name AS entity
ORDER BY relationship, entity
"""

cypher_df(query_6, document_id=document_id)

### Query for the UI

```
MATCH (d:Document {document_id: "<document_id>"})-[r1]->(e)<-[r2]-(other:Document)
WHERE other <> d
  AND (e:Technology OR e:BusinessTopic)
RETURN d, r1, e, r2, other
LIMIT 50
```